# Faster R-CNN — custom head / online ON / seed42 (run5)

Đây là cấu hình thứ 3 trong vòng đối chiếu seed42: standard/OFF, custom/OFF, custom/ON.
Mục tiêu: So ảnh hưởng **jitter online 0.1** với custom/OFF, không chọn model bằng test.

**Kết quả validation tham chiếu:** standard/OFF41.1759%; custom/OFF 42.7390% tại checkpointbest của mỗi run.
Đây chỉ là số đo 1 seed; run ON có thể tốt hơn hoặc thấp hơn. Sau run này chốt cấu hình bằng val,
rồi mới chạy thêm seeds 7,21,123,2026 cho cấu hình đã chọn.

**Thay đổi so với run4:** master online bật, jitter 0.1 có hiệu lực ở ảnh train và bbox.
Colorjitter, mosaic, mixup, hflip vẫn OFF. Val/test giữ nguyên; cache offline không sửa.
Custom head: Linear1024→512/ReLU/Dropout0.2 → class53/bbox212, giữ pretrained features.

**Giữ các điều kiện khác:** seed42, cachev1, 640px,20 epoch tối đa/patience10, 
SGDLR 0.005/momentum 0.9/weight_decay 0.0005, warmup 1 epoch + cosine, batch tổng 8/eval16, AMP/2 T4.
Bắt đầu lại từ pretrained COCO_V1; không load checkpoint run4 hoặc smoke.


## 1. Source và dependencies

Commit được pin theo các smoke đã qua. Giữ torch/torchvision Kaggle cung cấp và kiểm CUDA ở cell tiếp theo.


In [ ]:
import os, sys, subprocess, json, time, uuid
from pathlib import Path

REPO = Path('/kaggle/working/VN-Traffic-Sign-Detection')
EXPECTED_SHA = '17ad0eea18df9d414b39748a89eb39d9d06399df'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'faster-rcnn',
                    'https://github.com/tuan8p/VN-Traffic-Sign-Detection.git', str(REPO)], check=True)
sha = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Source:', sha)
assert sha == EXPECTED_SHA, 'Source đã thay đổi: đối chiếu commit trước khi chạy tiếp'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
# Giữ torch/torchvision do Kaggle cung cấp; không upgrade ngầm hoặc cài Pixi win-64 lên Linux.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO),
                'wandb==0.31.0', 'pycocotools==2.0.11'], check=True)

# Ghi cả xung đột package cài sẵn (vd google-adk/OpenTelemetry); không tự hạ package để che cảnh báo.
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
print('pip check exit:', pip_check.returncode)
print(pip_check.stdout or pip_check.stderr)


## 2. Hai GPU, CUDA NMS và mount cache


In [ ]:
import torch, torchvision, pycocotools
from torchvision.ops import nms

print('Python:', sys.version)
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)
print('CUDA:', torch.version.cuda, 'GPUs:', torch.cuda.device_count())
assert torch.cuda.is_available() and torch.cuda.device_count() == 2
assert torch.distributed.is_nccl_available(), 'Runtime chưa có NCCL'
assert hasattr(torch.amp, 'GradScaler'), 'Torch runtime chưa phù hợp API AMP pipeline'
for index in range(2):
    print(index, torch.cuda.get_device_name(index))
    device = f'cuda:{index}'
    kept = nms(torch.tensor([[0., 0., 10., 10.]], device=device),
               torch.tensor([.9], device=device), .5)
    assert kept.cpu().tolist() == [0]
    torch.cuda.synchronize(index)

candidates = sorted(p.parent for p in Path('/kaggle/input').rglob('data.yaml')
                    if p.parent.name == 'processed')
print('Processed roots:', candidates)
assert len(candidates) == 1, 'Chọn đúng một cache processed; không fallback sang raw/local'
DATA_ROOT = candidates[0]


## 3. Đăng nhập Forge bằng Kaggle Secret

Chỉ kiểm đăng nhập, không tạo run thử. Credentials được kế thừa qua environment cho torchrun; không ghi vào config/artifacts.


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

API_HOST = 'https://api.forge.coreweave.com'
ENTITY = 'HCMUT_IPCV'
PROJECT = 'BTL'

os.environ['WANDB_BASE_URL'] = API_HOST
os.environ['WANDB_MODE'] = 'online'
api_key = UserSecretsClient().get_secret('WANDB_API_KEY')
assert api_key.strip(), 'Kaggle Secret trống'
os.environ['WANDB_API_KEY'] = api_key

import wandb
print('wandb:', wandb.__version__)
assert wandb.login(key=api_key, host=API_HOST, relogin=True, verify=True, timeout=30)
del api_key
print('Đăng nhập thành công; bước tiếp theo sẽ kiểm quyền ghi project.')


## 4. Kiểm toàn bộ cache

Có thể mất nhiều phút để decode/hash mount. Cấu trúc hợp lệ không chứng minh split độc lập cảnh.


In [ ]:
from vn_tsd.config.resolve import resolve_config
from vn_tsd.pipelines.faster_rcnn.config import validate_config
from vn_tsd.pipelines.faster_rcnn.dataset import load_cache

cfg = validate_config(resolve_config(
    pipeline_yaml=REPO/'configs/pipelines/faster_rcnn.yaml',
    shared_yaml=REPO/'configs/shared.yaml', runtime_yaml=REPO/'configs/runtime/kaggle.yaml',
    overrides={'data': {'root': str(DATA_ROOT)}, 'train': {'require_wandb': True}, 'model': {'head': 'custom_mlp'}, 'project': {'wandb_base_url': API_HOST}}))
sets, identity = load_cache(cfg)
meta = json.loads((DATA_ROOT/'meta.json').read_text(encoding='utf-8'))
expected = {'train': (2660, 5743), 'val': (438, 1204), 'test': (371, 829)}
stems = {}
for split, dataset in sets.items():
    info = dataset.identity
    print(split, 'images', info['count'], 'boxes', info['annotations'],
          'small_boxes', info['small_boxes'], 'hash', info['sha256'])
    assert (info['count'], info['annotations']) == expected[split], 'Không khớp snapshotv1'
    assert info['count'] == meta['splits'][split]['n_images']
    assert info['annotations'] == meta['splits'][split]['n_boxes_out']
    listed = [Path(line.strip().replace('\\', '/')).stem for line in
              (DATA_ROOT/f'{split}_files.txt').read_text(encoding='utf-8').splitlines() if line.strip()]
    stems[split] = {r['path'].stem for r in dataset.records}
    assert len(listed) == len(set(listed)) and set(listed) == stems[split]
    assert info['small_boxes'] == 0, 'Báo Bình về box dưới policy8px; không tự lọc'
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    assert not stems[a] & stems[b], f'Trùng stem: {a}/{b}'
for split in ('val', 'test'):
    assert not any('_aug_' in stem for stem in stems[split])
    for stem in stems['train']:
        if '_aug_' in stem:
            assert stem.split('_aug_')[0] not in stems[split], 'Aug parent ở split khác'
seen = {label for r in sets['train'].records for label in r['labels']}
assert seen == set(range(1, 53)), 'Train thiếu lớp'
print('Full cache adapter check passed; chưa chứng minh độc lập cảnh/near-duplicate')


## 5. Tạo run mới và khóa cấu hình custom/ON

Không thay seed/batch/cache hoặc head để thử chất lượng trong run này. Cấu hình chung đã được resolve từrepo.

Master online ON chỉ bật jitter0.1 trong preset này; giữ tất cả phép khác như run4.


In [ ]:
import yaml

expected_hashes = {
    'train': 'fd15fcbf42da08bf274c99b267f1cae512463162a6f0b28ed84d31941c0dbd23',
    'val': 'e5139f467e918e47da0bad10c0068e5bcef193c45ab038bebd0007a475e6d9b3',
    'test': '3e02b6face375fcbcf441455970e515a4c5350e1638bd76c01ee6d7d62c6d7bb',
}
actual_hashes = {split: info['sha256'] for split, info in identity['splits'].items()}
print('Dataset split hashes:', actual_hashes)
assert actual_hashes == expected_hashes, 'Cache khác run OFF: dừng, đối chiếu version trước khi chạy'
assert sha == EXPECTED_SHA, 'Source khác run OFF: đối chiếu trước khi chạy'

RUN_NAME = 'frcnn_custom_s42_ep20_bs8_on_' + uuid.uuid4().hex[:8]
RUN_DIR = Path('/kaggle/working/frcnn_fulltrain') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
OVERRIDES = RUN_DIR/'launch_overrides.yaml'
overrides = {
    'seed': 42,
    'project': {'wandb_base_url': API_HOST},
    'train': {'epochs': 20, 'require_wandb': True},
    'online_preprocess': {'enabled': True},
    'model': {'pretrained': True, 'head': 'custom_mlp'},
    'eval': {'auto_eval_test': True},
}
OVERRIDES.write_text(yaml.safe_dump(overrides), encoding='utf-8')
(RUN_DIR/'cache_preflight.json').write_text(
    json.dumps({'source_sha': sha, 'kaggle_dataset':
        'zero001206/vn-traffic-sign-detection-preprocessed-data', 'version': 1,
        'scope': 'full custom-head/ON jitter0.1; cache v1 scene/frame limitations; test not for selection', 'dataset': identity},
        ensure_ascii=False, indent=2), encoding='utf-8')
with (RUN_DIR/'pip-freeze.txt').open('w') as versions:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=versions, check=True)
print('Run mới:', RUN_DIR)

(RUN_DIR/'pip-check.txt').write_text(pip_check.stdout + pip_check.stderr, encoding='utf-8')
(RUN_DIR/'experiment_notes.json').write_text(json.dumps({
    'experiment': 'custom_head_online_on_ablation',
    'seed': 42, 'epochs_max': 20, 'early_stopping_patience': 10,
    'split_policy': 'cache_v1_frozen_by_user_with_documented_scene_frame_overlap',
    'test_exposure': 'test metrics observed in prior smoke and standard-head baseline; not used for selection',
    'selection': 'validation mAP50-95 only',
    'supports_training_resume': False,
}, indent=2), encoding='utf-8')
# Giữ SDK runfiles trong cùng folder để export cùng checkpoint/log, không export /root/.netrc.
wandb_dir = RUN_DIR/'wandb_logs'
wandb_dir.mkdir()
os.environ['WANDB_DIR'] = str(wandb_dir)


## 6. Kiểm cấu hình cuối trước full train

Cell dừng nếu nhầm standard head, onlineOFF,2epoch hoặc sai Forgehost.


In [ ]:
from vn_tsd.pipelines.faster_rcnn.transforms import describe_effective
from vn_tsd.utils.io import load_yaml

launch_overrides = load_yaml(OVERRIDES)
launch_overrides['data'] = {'root': str(DATA_ROOT)}
launch_cfg = validate_config(resolve_config(
    pipeline_yaml=REPO/'configs/pipelines/faster_rcnn.yaml',
    shared_yaml=REPO/'configs/shared.yaml', runtime_yaml=REPO/'configs/runtime/kaggle.yaml',
    overrides=launch_overrides))
online = describe_effective(launch_cfg)['effective']
assert online['enabled'] is True and online['jitter'] == 0.1
assert all(online[k] is False for k in ('hflip', 'color_jitter', 'mosaic', 'mixup'))
assert launch_cfg['seed'] == 42
assert launch_cfg['train']['epochs'] == 20 and launch_cfg['train']['batch_size'] == 8
assert launch_cfg['eval']['batch_size'] == 16 and launch_cfg['train']['require_wandb'] is True
assert launch_cfg['model']['head'] == 'custom_mlp' and launch_cfg['model']['pretrained'] is True
assert launch_cfg['train']['amp'] and launch_cfg['runtime_cfg']['mixed_precision']
print('Config validated. Online effective:', online)
print('Run directory:', RUN_DIR)

assert launch_cfg['train']['early_stopping_patience'] == 10
assert launch_cfg['project']['wandb_base_url'] == API_HOST
assert launch_cfg['project']['wandb_entity'] == 'HCMUT_IPCV'
assert launch_cfg['project']['wandb_project'] == 'BTL'
assert os.environ['WANDB_MODE'] == 'online' and os.environ.get('WANDB_API_KEY')
print(json.dumps({
    'model_head': launch_cfg['model']['head'],
    'epochs_max': launch_cfg['train']['epochs'],
    'early_stopping_patience': launch_cfg['train']['early_stopping_patience'],
    'online_effective': online,
    'wandb': {'entity': ENTITY, 'project': PROJECT, 'host': API_HOST, 'required': True},
}, indent=2))

assert launch_cfg['model']['head_hidden_dim'] == 512
assert launch_cfg['model']['head_dropout'] == 0.2


## 7. Launch full training

Cell này dùngGPU và ghi trainingrun thật vàoForge. Đọc linkrun trongstdout để mởdashboard.
Theo dõi train_loss,val_loss,mAP50–95 vàLR; bestcheckpoint chọn từval. Test chỉ chạy sau khi reloadbest.
NếuOOM/worker/NCCL/W&Berror, giữ traceback đầu tiên vàconsole.log. Không tự giảm batch/disableW&B.
Nếu lỗi, có thể bỏ qua cáccellnghiệmthu rồi chạycellđónggói cuối để tải logs.


In [ ]:
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
    '-m', 'tools.run_train', '--pipeline', 'faster_rcnn',
    '--config', str(REPO/'configs/pipelines/faster_rcnn.yaml'),
    '--shared', str(REPO/'configs/shared.yaml'),
    '--runtime', str(REPO/'configs/runtime/kaggle.yaml'),
    '--data-root', str(DATA_ROOT), '--overrides', str(OVERRIDES),
    '--run-name', RUN_NAME, '--run-dir', str(RUN_DIR), '--online']
(RUN_DIR/'launch_command.json').write_text(json.dumps(command, indent=2), encoding='utf-8')
started = time.monotonic()
# Tee stdout+stderr để xem trực tiếp và giữ lỗi rank gốc cho buổi họp.
with (RUN_DIR/'console.log').open('w', encoding='utf-8') as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    exit_code = process.wait()
wall_seconds = time.monotonic() - started
(RUN_DIR/'launch_result.json').write_text(
    json.dumps({'exit_code': exit_code, 'wall_seconds': wall_seconds}), encoding='utf-8')
assert exit_code == 0, f'Giữ console.log và traceback đầu tiên; exit={exit_code}'


## 8. Kiểm kết quả training

Run có thể dừng trước20epoch doearlystop; không coi ít hơn20tựđộng là lỗi.


In [ ]:
import math
history = json.loads((RUN_DIR/'history.json').read_text())
metrics = json.loads((RUN_DIR/'metrics.json').read_text())
saved_cfg = yaml.safe_load((RUN_DIR/'resolved_config.yaml').read_text())
assert metrics['status'] == 'completed' and metrics['test_status'] == 'completed'
assert 1 <= len(history) <= 20
assert len(history) == 20 or history[-1]['patience_count'] >= 10
assert metrics['effective']['world_size'] == 2 and metrics['effective']['amp'] is True
assert metrics['effective']['online']['effective']['enabled'] is True
assert metrics['dataset_split_hashes'] == expected_hashes
assert saved_cfg['model']['head'] == 'custom_mlp'
assert saved_cfg['train']['epochs'] == 20 and saved_cfg['train']['require_wandb'] is True
assert saved_cfg['project']['wandb_base_url'] == API_HOST
assert saved_cfg['seed'] == 42
for row in history:
    assert math.isfinite(row['train_loss']) and math.isfinite(row['val_loss'])
    assert row['successful_updates'] > 0
    assert all(math.isfinite(v) for key in ('train_components','val_components') for v in row[key].values())
for name in ('checkpoints/checkpoint_best.pt', 'test_metrics.json', 'dataset_identity.json', 'run.zip'):
    assert (RUN_DIR/name).is_file(), name
print('Epochs thực chạy:', len(history), '| Best epoch:', metrics['best_epoch'])
print('Launcher minutes:', round(wall_seconds/60, 2))
print('Epoch seconds:', [round(row['seconds'], 2) for row in history])
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('Mở figures/ để xem curves và GT/pred. Không dùng test để chọn thay đổi config.')

assert saved_cfg['model']['head_hidden_dim'] == 512
assert saved_cfg['model']['head_dropout'] == 0.2

effective_online = metrics['effective']['online']['effective']
assert effective_online['jitter'] == 0.1
assert all(effective_online[k] is False for k in ('hflip', 'color_jitter', 'mosaic', 'mixup'))
assert saved_cfg['online_preprocess']['enabled'] is True
print('ONLINE ON thực tế:', effective_online)


## 9. Kiểm run Forge theo run ID

Không tìm bằng displayname: rename trên Forge không đổi runID. Cell lấy ID từ SDKfiles/console rồi đọc trạng thái;
nếu API chưa đồng bộ/không đọc được, giữ receipt và kiểm UI theo link. Không khởi tạo thêm run để kiểm run cũ.


In [ ]:
import re

# SDK run files và console URL chứa run ID ổn định dù tên hiển thị trên Forge đổi.
run_ids = set()
for path in (RUN_DIR/'wandb_logs').rglob('run-*.wandb'):
    match = re.fullmatch(r'run-([A-Za-z0-9]+)\.wandb', path.name)
    if match:
        run_ids.add(match.group(1))
console = (RUN_DIR/'console.log').read_text(encoding='utf-8', errors='replace')
run_ids.update(re.findall(
    re.escape(f'https://forge.coreweave.com/wandb/{ENTITY}/{PROJECT}/runs/') + r'([A-Za-z0-9]+)',
    console))
tracking_result = {
    'verification': 'check_ui', 'entity': ENTITY, 'project': PROJECT,
    'training_run_name': saved_cfg['train']['run_name'],
    'lookup_method': 'stable_run_id_from_sdk_or_console',
}
if len(run_ids) == 1:
    training_run_id = next(iter(run_ids))
    tracking_result.update(run_id=training_run_id,
        run_url=f'https://forge.coreweave.com/wandb/{ENTITY}/{PROJECT}/runs/{training_run_id}')
    try:
        api = wandb.Api(overrides={'base_url': API_HOST}, timeout=30)
        server_run = api.run(f'{ENTITY}/{PROJECT}/{training_run_id}')
        assert server_run.id == training_run_id
        tracking_result.update(server_state=server_run.state,
            display_name=server_run.name,
            verification='passed' if server_run.state == 'finished' else 'check_ui')
    except Exception as exc:
        tracking_result['error_type'] = type(exc).__name__
else:
    tracking_result['candidate_run_ids'] = sorted(run_ids)
(RUN_DIR/'tracking_verification.json').write_text(json.dumps(tracking_result, indent=2), encoding='utf-8')
print(json.dumps(tracking_result, indent=2))
print('Nếu check_ui: mở run_url hoặc link trong console.log và kiểm Finished/metrics. Không chạy train lại chỉ vì lookup lỗi.')


## 10. Export artifacts

Giữ toàn bộlog/checkpoint/metadata và SDKfiles trongbundle, không lồngrun.zip để tránhnhânđôi.
Download `*-artifacts.zip` và `.sha256`; SaveVersion để giữoutputs.
Local lưu `.agents/artifacts/<RUN_NAME>/` rồi gửiđườngdẫn review.
Fulltrain chưa được nghiệmthu chỉ vì cellcuốiđónggói thànhcông; kiểm cảmetrics/tracking/curves vàcheckpoint.


In [ ]:
from zipfile import ZipFile, ZIP_STORED
import hashlib

assert RUN_DIR.is_dir(), 'Chưa tạo run directory'
BUNDLE = Path('/kaggle/working') / f'{RUN_NAME}-artifacts.zip'
assert not BUNDLE.exists(), f'Bundle đã tồn tại: {BUNDLE}'
files = sorted(path for path in RUN_DIR.rglob('*') if path.is_file() and path.name != 'run.zip')
assert files, 'Run chưa có file để export'
with ZipFile(BUNDLE, 'x', compression=ZIP_STORED, allowZip64=True) as archive:
    for path in files:
        archive.write(path, arcname=f'{RUN_NAME}/{path.relative_to(RUN_DIR).as_posix()}')
with ZipFile(BUNDLE) as archive:
    assert archive.testzip() is None, 'Bundle CRC error'
sha256 = hashlib.sha256()
with BUNDLE.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
        sha256.update(block)
checksum_path = BUNDLE.with_suffix(BUNDLE.suffix + '.sha256')
checksum_path.write_text(f'{sha256.hexdigest()}  {BUNDLE.name}\n', encoding='utf-8')
print('Download:', BUNDLE)
print('Checksum:', checksum_path)
print('Local folder: .agents/artifacts/' + RUN_NAME + '/')
